## Generate dataset for multiplication only

- a from [0, 1000]
- b from [0, 1000]

Format is JSON:
```
[
    {
       input: [a, b],
       target: a * b
    }
    ...
    ...
]
```

In [17]:
import random
import json

dataset = []

unique_size = 20_000
unique_max_num = 1000

repeat_size = 5_000
repeat_max_num = 100

rand_dist_mu = 4.2
rand_dist_sigma = 1.0

for _ in range(repeat_size):
    a = random.randint(0, repeat_max_num)
    b = random.randint(0, repeat_max_num)
    target = a * b
    data = {
        'input': { 'a': str(a), 'b': str(b) },
        'target': str(target)
    }
    dataset.append(data)

for _ in range(unique_size):
    a = random.randint(0, unique_max_num)
    b = random.randint(0, unique_max_num)
    target = a * b
    data = {
        'input': { 'a': str(a), 'b': str(b) },
        'target': str(target)
    }
    dataset.append(data)

print(json.dumps(dataset[:5], indent=4))
print(f"completed dataset creation of size {len(dataset)}....")

[
    {
        "input": {
            "a": "4",
            "b": "1"
        },
        "target": "4"
    },
    {
        "input": {
            "a": "65",
            "b": "21"
        },
        "target": "1365"
    },
    {
        "input": {
            "a": "39",
            "b": "79"
        },
        "target": "3081"
    },
    {
        "input": {
            "a": "11",
            "b": "65"
        },
        "target": "715"
    },
    {
        "input": {
            "a": "13",
            "b": "70"
        },
        "target": "910"
    }
]
completed dataset creation of size 25000....


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from tokenizer import DigitTokenizer
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

CHECKPOINT_DIR = "/teamspace/studios/this_studio/SAIR-competition-modular-math/checkpoints"
DATA_DIR = "/teamspace/studios/this_studio/SAIR-competition-modular-math/data"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

rand_seed = 42

batch_size = 64
max_seq_len = 12 # very small context window...
base_lr = 1e-4

Training on device: cpu


In [18]:
tokenizer = DigitTokenizer()
eos_id = tokenizer.char_to_int["<eos>"]
pad_id = tokenizer.char_to_int["<pad>"]
tokenized_input = []

for sample in dataset:
    input_obj = sample["input"]
    str_tgt = sample["target"]
    str_a = input_obj['a']
    str_b = input_obj['b']
    
    a_token_id = tokenizer.encode(str_a)
    b_token_id = tokenizer.encode(str_b)
    tgt_token_id = tokenizer.encode(str_tgt)
    tgt_token_id.append(eos_id)

    if len(a_token_id) < max_seq_len:
        a_token_id = a_token_id + [pad_id] * (max_seq_len - len(a_token_id))
    else:
        a_token_id = a_token_id[:max_seq_len]
    
    if len(b_token_id) < max_seq_len:
        b_token_id = b_token_id + [pad_id] * (max_seq_len - len(b_token_id))
    else:
        b_token_id = b_token_id[:max_seq_len]
    
    if len(tgt_token_id) < max_seq_len:
        tgt_token_id = tgt_token_id + [pad_id] * (max_seq_len - len(tgt_token_id))
    else:
        tgt_token_id = tgt_token_id[:max_seq_len]

    tokenized_input.append([a_token_id, b_token_id, tgt_token_id])


dataset_tensor = torch.tensor(tokenized_input, dtype=torch.long)

print(f"Dataset securely loaded in-memory. Matrix shape: {dataset_tensor.shape}")
print(f"The <eos> token id is: {tokenizer.char_to_int['<eos>']}")
print(dataset_tensor[:3])

Dataset securely loaded in-memory. Matrix shape: torch.Size([25000, 3, 12])
The <eos> token id is: 11
tensor([[[ 4, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10],
         [ 1, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10],
         [ 4, 11, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10]],

        [[ 6,  5, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10],
         [ 2,  1, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10],
         [ 1,  3,  6,  5, 11, 10, 10, 10, 10, 10, 10, 10]],

        [[ 3,  9, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10],
         [ 7,  9, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10],
         [ 3,  0,  8,  1, 11, 10, 10, 10, 10, 10, 10, 10]]])
